In [ ]:
#!pip install konlpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.1/494.1 kB 29.0 MB/s eta 0:00:00


In [ ]:
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import urllib.request
from konlpy.tag import Okt
from tqdm import tqdm
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
urllib.request.urlretrieve("https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt", filename="ratings_train.txt")
urllib.request.urlretrieve("https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt", filename="ratings_test.txt")

('ratings_test.txt', <http.client.HTTPMessage at 0x785ae5746090>)

In [ ]:
train_data = pd.read_csv('ratings_train.txt', sep='\t')
test_data = pd.read_csv('ratings_test.txt', sep='\t')

print(train_data.head())
print(test_data.head())

         id                                           document  label
0   9976970                                아 더빙.. 진짜 짜증나네요 목소리      0
1   3819312                  흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나      1
2  10265843                                  너무재밓었다그래서보는것을추천한다      0
3   9045019                      교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정      0
4   6483659  사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...      1
        id                                           document  label
0  6270596                                                굳 ㅋ      1
1  9274899                               GDNTOPCLASSINTHECLUB      0
2  8544678             뭐야 이 평점들은.... 나쁘진 않지만 10점 짜리는 더더욱 아니잖아      0
3  6825595                   지루하지는 않은데 완전 막장임... 돈주고 보기에는....      0
4  6723715  3D만 아니었어도 별 다섯 개 줬을텐데.. 왜 3D로 나와서 제 심기를 불편하게 하죠??      0


In [ ]:
stopwords = []

with open("Kor_StopWords.txt", "r", encoding="utf-8") as f:
    for line in f:
        stopwords.extend(line.strip().split(','))

print('한국어 stop words 갯수:', len(stopwords))
print(stopwords[:20])

한국어 stop words 갯수: 251
['아', ' 휴', ' 아이구', ' 아이쿠', ' 아이고', ' 어', ' 나', ' 우리', ' 저희', ' 따라', ' 의해', ' 을', ' 를', ' 에', ' 의', ' 가', ' 으로', ' 로', ' 에게', ' 뿐이다']


In [ ]:
train_data.info()
test_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        150000 non-null  int64 
 1   document  149995 non-null  object
 2   label     150000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 3.4+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        50000 non-null  int64 
 1   document  49997 non-null  object
 2   label     50000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 1.1+ MB


In [ ]:
train_data = train_data.dropna(subset=['document'])
test_data = test_data.dropna(subset=['document'])

In [ ]:
train_data.info()
test_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 149995 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        149995 non-null  int64 
 1   document  149995 non-null  object
 2   label     149995 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 4.6+ MB
<class 'pandas.core.frame.DataFrame'>
Index: 49997 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        49997 non-null  int64 
 1   document  49997 non-null  object
 2   label     49997 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 1.5+ MB


In [ ]:
from konlpy.tag import Hannanum
hannanum = Hannanum()

def tokenize_text(text):
  word_token = hannanum.morphs(text)
  return (word_token)

tokenize_text(train_data.document[1])

['흠', '.', '..', '포스터보고', '초딩영화줄', '....', '오버연기', '조차', '가볍', '지', '않', '구나']

In [ ]:
import re

sent_word = []

# Iterate over the actual index of the DataFrame
for i in train_data.index:
    tokens = tokenize_text(train_data.document[i])
    cleaned_tokens = [re.sub(r"[^\sㄱ-ㅎㅏ-ㅣ가-힣]", "", word) for word in tokens if word not in stopwords]
    sent_word.append(cleaned_tokens)

print(sent_word[:5])  # 첫 5개 항목만 출력하여 결과 확인

[['더빙', '', '진짜', '짜증나', '네', '요', '목소리'], ['흠', '', '', '포스터보고', '초딩영화줄', '', '오버연기', '조차', '가볍', '지', '않', '구나'], ['너무재밓었다그래서보는것을추천한다'], ['교도소', '이야기구먼', '', '솔직히', '재미', '는', '없', '다', '', '평점', '조정'], ['사이몬페그', '의', '익살', '스런', '연기', '가', '돋보이', '었던', '영화', '', '스파이더맨', '에서', '늙', '어', '보이', '기', '만', '하', '었던', '커스틴', '던스트', '가', '너무나', '도', '이쁘', '어', '보이', '었다']]


In [ ]:
sent_word

[['더빙', '', '진짜', '짜증나', '네', '요', '목소리'],
 ['흠', '', '', '포스터보고', '초딩영화줄', '', '오버연기', '조차', '가볍', '지', '않', '구나'],
 ['너무재밓었다그래서보는것을추천한다'],
 ['교도소', '이야기구먼', '', '솔직히', '재미', '는', '없', '다', '', '평점', '조정'],
 ['사이몬페그',
  '의',
  '익살',
  '스런',
  '연기',
  '가',
  '돋보이',
  '었던',
  '영화',
  '',
  '스파이더맨',
  '에서',
  '늙',
  '어',
  '보이',
  '기',
  '만',
  '하',
  '었던',
  '커스틴',
  '던스트',
  '가',
  '너무나',
  '도',
  '이쁘',
  '어',
  '보이',
  '었다'],
 ['막',
  '걸음마',
  '떼',
  'ㄴ',
  '세',
  '부터',
  '초등학교',
  '학년생',
  '이',
  'ㄴ',
  '살용영화',
  '',
  'ㅋㅋㅋ',
  '',
  '별반개',
  '도',
  '아깝',
  '음',
  ''],
 ['원작', '의', '긴장감', '을', '제대로', '살리', '어', '내', '지', '못하', '었다', ''],
 ['별',
  '반개',
  '도',
  '아깝',
  '다',
  '욕나온다',
  '이응경',
  '길용우',
  '연기생활이몇년',
  '이',
  'ㄴ지',
  '',
  '정말',
  '발로해',
  '도',
  '그것보단',
  '낫겟다',
  '납치',
  '',
  '감금만반복반복',
  '',
  '이드라마',
  '는',
  '가족도',
  '없',
  '다',
  '연기못하는사람만모엿',
  '네'],
 ['액션', '이', '없', '는데', '도', '재미', '있', '는', '몇안되', '는', '영화'],
 ['왜케',
  '평점',
  '이',
  '낮',
  '은',
  '것',
  '이'

In [ ]:
num_size = 10000
tokenizer = Tokenizer(num_words=num_size)
tokenizer.fit_on_texts(train_data["document"])

In [ ]:
import numpy as np

def vectorize_sequences_2(sequences,dimension=10000):  #dimension은 feature의 차원수를 의미
  # results = np.zeros((행, 열))
  results = np.zeros((len(sequences), dimension)) #상수는 프로그래밍할 때 지양하자
  for i, sequence in enumerate(sequences): #i에는 인덱스, sequence에는 value가 옴(에뮤레이트(enmerate) == 열거)
    for j in sequence: #j에는 word_index가 들어옴(sequence = [1,14,168,40,...] 여기서 1, 14 등의 값이 들어온다.)
      results[i, j] = 1.
  return results

x_train = vectorize_sequences_2(train_data)

IndexError: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices